In [1]:
from datetime import datetime

import pandas as pd
import yfinance.screener.query

# import pandas as pd

#IMPORTS

from functions import *





In [2]:
#INPUTS

target_ticker = "ADS.DE"

benchmark = "URTH"
start_date = "2020-12-31"
end_date = "2026-08-01"
interval = "1wk"
return_calc = "linear" #linear / log
beta_adjustment = "blume" #blume / vasicek / none
peer_group_beta_method = "median" #average / median
rf_lookback_months = 1 #from valuation date or latest available data
equity_risk_premium = 0.055
size_premium = 0.015
comp_spec_risk_premium = 0.00
target_longt_sp_rating = "A"
debt_spread_lookback_months = 1 #from valuation date or latest available data

peer_group = ["NKE", "PUM.DE", "ONON", "DECK", "CROX"]


# Ce = rf + beta * mrp + sp +csrp
# Cd = rf + debt spread

In [3]:
#CLOSE_DATA_COLLECTION

ticker_package = peer_group + [benchmark]
data_package = download_data(tickers= ticker_package, start_date= start_date, end_date= end_date, interval= interval)
save_data(data = data_package, benchmark= benchmark, peer_group= peer_group, start_date= start_date, end_date= end_date)
close_data = extract_col(data = data_package, field= "Close")



[*********************100%***********************]  6 of 6 completed


In [4]:
#BASIC_DATA_CLEANING
close_data = basic_cleaning(close_data=close_data)

In [5]:
#LOG_RETURN_CALC
return_data = log_return_calc(return_calc=return_calc, close_data= close_data)

In [6]:
#BETA_REGRESSION
beta_results = beta_regression(peer_group=peer_group, return_data=return_data, benchmark=benchmark)

In [7]:
beta_adj = beta_adjustments(beta_results)

In [8]:
beta_res = append_d_e_ratio(beta_results=beta_adj, end_date= end_date)

In [9]:
beta_results = append_tax_rates(beta_results=beta_res)

In [10]:
peer_group_beta = unlevered_beta(beta_adjustment=beta_adjustment, beta_results=beta_results, peer_group_beta_method=peer_group_beta_method)

In [11]:
target_levered_beta = get_target_levered(target=target_ticker, end_date=end_date, peer_group_beta=peer_group_beta)

In [12]:
target_rf_rate = (target_rf(target=target_ticker, BASE_STR_1=BASE_STRING_1, BASE_STR_2=BASE_STRING_2,start_date=start_date, end_date=end_date, rf_lookback=rf_lookback_months)) /100

In [13]:
spread_series = get_rating_spread_series(rating_spread_series=RATING_SPREAD_SERIES, start_date=start_date, end_date=end_date)

In [14]:
interpolated_series = interpolate_spreads(spread_series)

In [15]:
target_spread = get_target_spread(interpolated_series, target_longt_sp_rating, lookback=debt_spread_lookback_months)

In [18]:
close_data

Ticker,CROX,DECK,NKE,ONON,PUM.DE,URTH
Date,,,,,,
2021-09-13,155.179993,72.498337,144.238388,38.950001,93.724205,120.297226
2021-09-20,156.300003,64.791664,137.940292,36.020000,92.077644,120.685379
2021-09-27,141.130005,60.911667,135.607315,30.500000,90.570610,117.829544
2021-10-04,130.399994,59.973331,140.605225,30.070000,91.631104,118.485741
2021-10-11,137.190002,59.711666,145.704544,29.719999,94.468422,121.073555
...,...,...,...,...,...,...
2026-06-29,125.279999,104.690002,44.090000,36.830002,26.889999,202.649994
2026-07-06,132.779999,105.989998,44.369999,38.540001,28.180000,204.630005
2026-07-13,137.130005,106.489998,43.759998,37.200001,28.680000,201.899994


In [17]:
wacc = get_wacc(target_ticker=target_ticker, end_date=end_date, rf=target_rf_rate, debt_spread=target_spread, relevered_beta=target_levered_beta, erp=equity_risk_premium, sp=size_premium, csrp=comp_spec_risk_premium)

print(wacc)

0.0952852976766341
